In [16]:
# INVERTED INDEX (Manning-style, in-memory) for text files
# ================================================================
#
# What this code does:
#   1) Looks for all .txt files in the SAME directory as the notebook
#   2) Assigns a document ID to each file (D1, D2, D3, ...)
#   3) Reads and tokenizes the text of each file
#   4) Builds an inverted index in memory
#   5) Prints:
#        - the document ID to filename mapping
#        - the inverted index on screen
#
# Connection to Manning:
#   In Chapter 1, the book moves from the term-document incidence matrix
#   to the inverted index because the matrix is inefficient for large
#   collections. Instead of storing for each term a full row of 0s and 1s,
#   we store only the list of document IDs in which the term appears.
#
#   Example:
#       retrieval -> [1, 2, 3]
#       machine   -> [3]
#
#   These lists are called postings lists.
#
# Assumption:
#   All .txt files are in the current working directory
#   (usually the same folder as the Jupyter notebook).
# ================================================================

# -----------------------------
# Imports
# -----------------------------

import re
from pathlib import Path
from typing import Dict, List, Set, Tuple

In [17]:
# -----------------------------
# Tokenization pattern
# -----------------------------
#
# This regular expression finds "words" made of letters and digits.
# It also allows an optional apostrophe part, so words like "don't"
# can be treated as a single token.
#
# Examples matched:
#   information
#   retrieval
#   model1
#   don't
#
# Examples not specially handled:
#   punctuation like commas, dots, etc.
#   Greek letters (this regex is mainly for English/alphanumeric text)
#
WORD_RE = re.compile(r"[A-Za-z0-9]+(?:'[A-Za-z0-9]+)?")

# -----------------------------
# Function: tokenize
# -----------------------------
def tokenize(text: str) -> List[str]:
    """
    Convert a raw text string into a list of normalized tokens.

    Steps:
      1) Find all regex matches in the text
      2) Extract the matched text
      3) Convert each token to lowercase

    Why lowercase?
      Because in IR we usually normalize case so that:
          Information
          information
          INFORMATION
      are all treated as the same term.

    Parameters:
      text: the raw text content of a document

    Returns:
      A list of tokens, in the order they appear in the text
    """
    return [match.group(0).lower() for match in WORD_RE.finditer(text)]
    

In [18]:
# -----------------------------
# Function: load_documents
# -----------------------------
def load_documents() -> List[Path]:
    """
    Find all .txt files in the current directory and return them sorted.

    Why sort them?
      Because we want stable and predictable document IDs.
      If the files are always sorted the same way, then:
          D1, D2, D3, ...
      will always refer to the same files across runs.

    Returns:
      A sorted list of Path objects for .txt files

    Raises:
      ValueError if no .txt files are found
    """

    # Path(".") means "the current working directory"
    current_folder = Path(".")

    # current_folder.glob("*.txt") finds all files whose names end in .txt
    # We additionally keep only real files with p.is_file()
    # sorted(...) ensures deterministic ordering
    files = sorted([p for p in current_folder.glob("*.txt") if p.is_file()])

    if not files:
        raise ValueError("No .txt files found in the current directory.")

    return files

In [19]:
# -----------------------------
# Function: build_inverted_index
# -----------------------------
def build_inverted_index(files: List[Path]) -> Dict[str, List[int]]:
    """
    Build an inverted index in memory.

    Inverted index structure:
      term -> postings list

    A postings list is the list of document IDs in which the term appears.

    Example:
      {
          "information": [1, 2, 3],
          "machine": [3],
          "retrieval": [1, 2, 3]
      }

    Important design choice:
      We use a set of terms per document before inserting into the index.
      This ensures that each document ID is inserted only once per term.

      For example, if a document contains:
          "information information retrieval"
      then for a basic inverted index we want:
          information -> [doc_id]
      not:
          information -> [doc_id, doc_id]

    Parameters:
      files: sorted list of Path objects

    Returns:
      A dictionary mapping term -> sorted postings list
    """

    # Create an empty dictionary for the inverted index.
    #
    # Keys:
    #   terms (strings)
    #
    # Values:
    #   postings lists (lists of document IDs)
    #
    inverted_index: Dict[str, List[int]] = {}

    # Loop through files and assign document IDs starting at 1
    for doc_id, path in enumerate(files, start=1):

        # Read the file content as text.
        # utf-8 is the most common text encoding.
        # errors="ignore" avoids crashing if a file contains bad bytes.
        text = path.read_text(encoding="utf-8", errors="ignore")

        # Tokenize the text into normalized terms
        tokens = tokenize(text)

        # Convert tokens to a set so each term appears only once per document
        unique_terms_in_doc: Set[str] = set(tokens)

        # For each unique term found in this document,
        # insert the document ID into that term's postings list
        for term in unique_terms_in_doc:

            # If the term is not already in the dictionary,
            # create a new empty postings list for it
            if term not in inverted_index:
                inverted_index[term] = []

            # Append the current doc_id to the postings list
            inverted_index[term].append(doc_id)

    # Finally, sort the dictionary by term so printing is easier to read
    #
    # sorted(inverted_index.items()) returns pairs like:
    #   ("and", [1, 3])
    #   ("information", [1, 2, 3])
    #
    # dict(...) turns those sorted pairs back into a dictionary.
    #
    inverted_index = dict(sorted(inverted_index.items()))

    return inverted_index


In [ ]:
def parse_query(query: str):
    
    query = query.strip()

    if " AND " in query and " OR " in query:
        raise ValueError("Δεν υποστηρίζονται μικτά queries.")

    if " AND " in query:
        terms = [t.lower().strip() for t in query.split(" AND ")]
        return "AND", terms

    elif " OR " in query:
        terms = [t.lower().strip() for t in query.split(" OR ")]
        return "OR", terms

    else:
        raise ValueError("Το query πρέπει να περιέχει AND ή OR.")


def and_query(terms, inverted_index):
    postings = [inverted_index.get(t, []) for t in terms]

    # 🔥 αυτό είναι το optimization που ζητά η εργασία
    postings.sort(key=len)

    result = set(postings[0])

    for p in postings[1:]:
        result = result.intersection(p)

    return sorted(result)


def or_query(terms, inverted_index):
    result = set()

    for t in terms:
        result = result.union(inverted_index.get(t, []))

    return sorted(result)


def evaluate_query(query, inverted_index):
    operator, terms = parse_query(query)

    if operator == "AND":
        return and_query(terms, inverted_index)

    elif operator == "OR":
        return or_query(terms, inverted_index)
    

In [ ]:
def parse_query(query: str):
    query = query.strip()

    if " AND " in query and " OR " in query:
        raise ValueError("Δεν υποστηρίζονται μικτά queries.")

    if " AND " in query:
        terms = [t.lower().strip() for t in query.split(" AND ")]
        return "AND", terms

    elif " OR " in query:
        terms = [t.lower().strip() for t in query.split(" OR ")]
        return "OR", terms

    else:
        raise ValueError("Το query πρέπει να περιέχει AND ή OR.")

In [ ]:
def and_query(terms, inverted_index):
    postings = [inverted_index.get(t, []) for t in terms]

    # 🔥 optimization
    postings.sort(key=len)

    result = set(postings[0])

    for p in postings[1:]:
        result = result.intersection(p)

    return sorted(result)

In [ ]:
def or_query(terms, inverted_index):
    result = set()

    for t in terms:
        result = result.union(inverted_index.get(t, []))

    return sorted(result)

In [ ]:
def evaluate_query(query, inverted_index):
    operator, terms = parse_query(query)

    if operator == "AND":
        return and_query(terms, inverted_index)

    elif operator == "OR":
        return or_query(terms, inverted_index)

In [ ]:
def query_loop(inverted_index, files):
    print("\nBoolean Query Mode")
    print("Γράψε 'exit' για έξοδο\n")

    while True:
        query = input("Query: ")

        if query.lower() == "exit":
            break

        try:
            result_ids = evaluate_query(query, inverted_index)

            filenames = [files[i-1].name for i in result_ids]

            print("\n--- RESULT ---")
            print("Query:", query)
            print("Document IDs:", result_ids)
            print("Filenames:", filenames)
            print("--------------\n")

        except Exception as e:
            print("Error:", e)

In [20]:
# -----------------------------
# Function: print_document_mapping
# -----------------------------
def print_document_mapping(files: List[Path]) -> None:
    """
    Print the mapping from document IDs to filenames.

    Example:
      D1: doc1.txt
      D2: doc2.txt
      D3: doc3.txt
    """
    print("DOCUMENT ID MAPPING")
    print("=" * 50)

    for doc_id, path in enumerate(files, start=1):
        print(f"D{doc_id}: {path.name}")

    print()  # blank line after the mapping


In [21]:
# -----------------------------
# Function: print_inverted_index
# -----------------------------
def print_inverted_index(inverted_index: Dict[str, List[int]]) -> None:
    """
    Print the inverted index on screen in a readable format.

    Example:
      information -> [1, 2, 3]
      machine     -> [3]
      retrieval   -> [1, 2, 3]

    We align terms so the output looks cleaner.
    """

    print("INVERTED INDEX")
    print("=" * 50)

    # Find the longest term length so we can align the arrows nicely
    term_width = max((len(term) for term in inverted_index), default=4)

    # Print each term and its postings list
    for term, postings_list in inverted_index.items():
        print(f"{term.ljust(term_width)} -> {postings_list}")


In [22]:
# -----------------------------
# Function: intersect_postings
# -----------------------------
def intersect_postings(postings1: List[int], postings2: List[int]) -> List[int]:
    """
    Intersect two sorted postings lists using the classic two-pointer
    merge algorithm described in Manning.

    Why this works:
      Because both postings lists are sorted in ascending docID order.
      We compare the current docIDs from each list:

      - If they are equal:
            the docID appears in both lists, so it belongs in the result.
            Then advance both pointers.
      - If postings1[i] < postings2[j]:
            move pointer i forward, because postings1[i] cannot appear later
            in postings2 (which is already at a larger value).
      - If postings2[j] < postings1[i]:
            move pointer j forward for the symmetric reason.

    Example:
      postings1 = [1, 2, 4, 8]
      postings2 = [2, 4, 6, 8]

      result = [2, 4, 8]

    Parameters:
      postings1: first sorted postings list
      postings2: second sorted postings list

    Returns:
      A new sorted list containing only the docIDs that appear in both lists
    """

    # i points to the current position in postings1
    i = 0

    # j points to the current position in postings2
    j = 0

    # This list will store the intersection result
    answer: List[int] = []

    # Continue while neither pointer has reached the end of its list
    while i < len(postings1) and j < len(postings2):

        # Case 1: same docID in both postings lists
        if postings1[i] == postings2[j]:
            answer.append(postings1[i])
            i += 1
            j += 1

        # Case 2: docID in postings1 is smaller, so move i
        elif postings1[i] < postings2[j]:
            i += 1

        # Case 3: docID in postings2 is smaller, so move j
        else:
            j += 1

    return answer

In [ ]:
# -----------------------------
# Function: parse_and_query
# -----------------------------
def parse_and_query(query: str) -> Tuple[str, str]:
    """
    Parse a user query of the form:
        term1 AND term2

    The function is intentionally simple and supports only this exact
    two-term Boolean AND format.

    Steps:
      1) Remove extra spaces at the ends
      2) Split on the word AND
      3) Normalize both terms to lowercase

    Example:
      "Information AND Retrieval"
      -> ("information", "retrieval")

    Parameters:
      query: raw user query string

    Returns:
      A tuple (term1, term2)

    Raises:
      ValueError if the query is not in the expected format
    """

    # Remove leading/trailing whitespace
    query = query.strip()

    # Split only on the exact Boolean operator AND with spaces around it.
    # This keeps the assignment simple and matches the requested form.
    parts = query.split(" AND ")

    if len(parts) != 2:
        raise ValueError("Query must be exactly in the form: term1 AND term2")

    # Normalize the two terms:
    # - strip extra spaces
    # - lowercase
    term1 = parts[0].strip().lower()
    term2 = parts[1].strip().lower()

    if not term1 or not term2:
        raise ValueError("Both terms must be non-empty.")

    return term1, term2

In [23]:
# -----------------------------
# Function: evaluate_and_query
# -----------------------------
def evaluate_and_query(query: str, inverted_index: Dict[str, List[int]]) -> List[int]:
    """
    Evaluate a query of the form:
        term1 AND term2

    Steps:
      1) Parse the query into two normalized terms
      2) Retrieve each term's postings list from the inverted index
      3) Intersect the two postings lists using Manning's merge algorithm

    Important:
      If a term does not exist in the index, its postings list is empty.
      Intersecting with an empty list gives an empty result.

    Parameters:
      query: the raw query string
      inverted_index: the in-memory inverted index

    Returns:
      A list of matching docIDs
    """
    term1, term2 = parse_and_query(query)

    # Get postings lists; if a term is missing, use empty list
    postings1 = inverted_index.get(term1, [])
    postings2 = inverted_index.get(term2, [])

    # Merge/intersect the two postings lists
    result = intersect_postings(postings1, postings2)

    return result

In [24]:
# -----------------------------
# Function: print_query_result
# -----------------------------
def print_query_result(query: str, result_doc_ids: List[int], files: List[Path]) -> None:
    """
    Print the result of the AND query.

    If matching docIDs are found, also print the corresponding filenames.
    """
    print("QUERY")
    print("=" * 60)
    print(query)
    print()

    print("MATCHING DOCUMENT IDs")
    print("=" * 60)
    print(result_doc_ids)
    print()

    if result_doc_ids:
        print("MATCHING FILENAMES")
        print("=" * 60)
        for doc_id in result_doc_ids:
            # doc_id starts at 1, but Python lists are 0-based
            print(f"D{doc_id}: {files[doc_id - 1].name}")
    else:
        print("No documents matched the query.")


In [25]:
# execution 

In [26]:
files = load_documents()
print_document_mapping(files)

inverted_index = build_inverted_index(files)
print_inverted_index(inverted_index)

DOCUMENT ID MAPPING
D1: Document1.txt
D2: Document10.txt
D3: Document2.txt
D4: Document3.txt
D5: Document4.txt
D6: Document5.txt
D7: Document6.txt
D8: Document7.txt
D9: Document8.txt
D10: Document9.txt

INVERTED INDEX
a              -> [1, 3, 4, 5, 6, 7, 8, 9, 10]
abducted       -> [6]
aboard         -> [1, 3, 4, 5]
about          -> [1, 4, 7]
accept         -> [6]
acting         -> [1]
advancing      -> [2]
after          -> [2, 3, 7, 9, 10]
against        -> [3]
agent          -> [6]
ago            -> [6]
air            -> [3]
alex           -> [1]
alive          -> [5]
all            -> [4, 6, 7]
alliance       -> [1, 7]
almost         -> [4]
alone          -> [1]
also           -> [9, 10]
ambushed       -> [9]
amos           -> [1, 2]
an             -> [3, 4, 5, 6, 7, 9]
and            -> [1, 2, 4, 5, 6, 7, 8, 9, 10]
anderson       -> [6]
antenna        -> [3]
anubis         -> [9, 10]
apartment      -> [3, 6]
are            -> [2, 4, 9, 10]
arrive         -> [7]
arrives        -> 

In [ ]:
def parse_query(query: str):
    def parse_query(query: str):
    query = query.strip()

    if " AND " in query and " OR " in query:
        raise ValueError("Δεν υποστηρίζονται μικτά queries.")

    if " AND " in query:
        terms = [t.lower().strip() for t in query.split(" AND ")]
        return "AND", terms

    elif " OR " in query:
        terms = [t.lower().strip() for t in query.split(" OR ")]
        return "OR", terms

    else:
        raise ValueError("Το query πρέπει να περιέχει AND ή OR.")


def and_query(terms, inverted_index):
    postings = [inverted_index.get(t, []) for t in terms]

    postings.sort(key=len)  # 🔥 optimization

    result = set(postings[0])

    for p in postings[1:]:
        result = result.intersection(p)

    return sorted(result)


def or_query(terms, inverted_index):
    result = set()

    for t in terms:
        result = result.union(inverted_index.get(t, []))

    return sorted(result)


def evaluate_query(query, inverted_index):
    operator, terms = parse_query(query)

    if operator == "AND":
        return and_query(terms, inverted_index)

    elif operator == "OR":
        return or_query(terms, inverted_index)

In [ ]:
from pathlib import Path

folder = Path()
files = sorted(folder.glob("*.txt"))

inverted_index = build_inverted_index(files)

print("Index built successfully!")

In [ ]:
queries = [
    queries = [
    "information AND retrieval",
    "information OR machine",
    "retrieval AND system AND information"
]

for q in queries:
    result_ids = evaluate_query(q, inverted_index)
    filenames = [files[i-1].name for i in result_ids]

    print("\n--- RESULT ---")
    print("Query:", q)
    print("Document IDs:", result_ids)
    print("Filenames:", filenames)
]

In [28]:
# Ask the user to enter a Boolean AND query
# Example:
#   information AND retrieval
user_query = input("Enter a query in the form: term1 AND term2\n> ")

try:
    result = evaluate_and_query(user_query, inverted_index)
    print()
    print_query_result(user_query, result, files)
except ValueError as e:
    print()
    print(f"Invalid query: {e}")

Enter a query in the form: term1 AND term2
>  rocinante AND ship



QUERY
rocinante AND ship

MATCHING DOCUMENT IDs
[2, 6, 8, 9, 10]

MATCHING FILENAMES
D2: Document10.txt
D6: Document5.txt
D8: Document7.txt
D9: Document8.txt
D10: Document9.txt
